In [52]:
import warnings
warnings.filterwarnings('ignore')

In [53]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

In [54]:
res = pd.read_csv("data/1976-2024-house.tab", encoding='utf-8')
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
0,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,BILL DAVENPORT,DEMOCRAT,False,TOTAL,58906,157170,False,20250910,False
1,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,JACK EDWARDS,REPUBLICAN,False,TOTAL,98257,157170,False,20250910,False
2,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,WRITEIN,NaN,True,TOTAL,7,157170,False,20250910,False
3,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,J CAROLE KEAHEY,DEMOCRAT,False,TOTAL,66288,156362,False,20250910,False
4,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,"WILLIAM L \""BILL\"" DICKINSON",REPUBLICAN,False,TOTAL,90069,156362,False,20250910,False


In [55]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'runoff', 'special', 'candidate',
       'party', 'writein', 'mode', 'candidatevotes', 'totalvotes',
       'unofficial', 'version', 'fusion_ticket'], dtype=object)

In [56]:
res.shape

(33805, 20)

In [57]:
ak_res_rcv = pd.read_csv('data/alaska_rcv_maximum_round_house_results.csv')
ak_res_rcv.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
0,2022,ALASKA,AK,2,94,81,US HOUSE,0,GEN,False,False,MARY SATTLER PELTOLA,DEMOCRAT,False,TOTAL,137263,249734,False,128553,False
1,2022,ALASKA,AK,2,94,81,US HOUSE,0,GEN,False,False,SARAH PALIN,REPUBLICAN,False,TOTAL,112471,249734,False,67866,False
2,2024,ALASKA,AK,2,94,81,US HOUSE,0,GEN,False,False,MARY SATTLER PELTOLA,DEMOCRAT,False,TOTAL,156985,321846,False,152828,False
3,2024,ALASKA,AK,2,94,81,US HOUSE,0,GEN,False,False,NICK BEGICH,REPUBLICAN,False,TOTAL,164861,321846,False,159550,False


In [58]:
# 2022 and 2024 Alaska at-large district results are plurality (first round) vote, not maximum round RCV
# We want to train with maximum round
# Replace with max round results
# File is manually inputted; results from Wikipedia

ak_mask = ((res['state'] == 'ALASKA') &
          (res['year'] >= 2022))

res = res[~ak_mask]
res = pd.concat([res, ak_res_rcv], axis=0)

In [59]:
# Fix text encoding errors
res['candidate'] = res['candidate'].map(ftfy.fix_text)

In [60]:
cutoff_year = 2014
res = res[(res['year'] >= cutoff_year - 2) & (res['stage'] == 'GEN') & (res['mode'] == 'TOTAL')]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
24053,2012,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,JO BONNER,REPUBLICAN,False,TOTAL,196374,200676,False,20250910,False
24054,2012,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,WRITEIN,NaN,True,TOTAL,4302,200676,False,20250910,False
24055,2012,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,180591,283953,False,20250910,False
24056,2012,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,THERESE FORD,DEMOCRAT,False,TOTAL,103092,283953,False,20250910,False
24057,2012,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,WRITEIN,NaN,True,TOTAL,270,283953,False,20250910,False


In [61]:
res = res[~(res['candidate'] == 'DAVID ALEXANDER')]

In [62]:
# Handling fusion voting
resdem = res[res['party'] == 'DEMOCRAT']
resrep = res[res['party'] == 'REPUBLICAN']
res3rd = res[~res['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]
res = pd.concat([resdem, resrep, res3rd], axis=0)

res_fusion = res.copy() #res[res['fusion_ticket']]

res_fusion = res_fusion.groupby(['candidate', 'year', 'state', 'state_po', 'state_fips',
                                'state_cen', 'state_ic', 'office', 'district']).agg({
    'stage':'first', 'runoff': 'first', 'special': 'first', 'party': 'first', 'writein': 'first', 'mode': 'first',
    'totalvotes': 'first', 'unofficial': 'first', 'version': 'first',
    'candidatevotes': 'sum'})

# res_notfusion = res[~res['fusion_ticket']]
res = res_fusion.reset_index() # pd.concat([res_fusion.reset_index(), res_notfusion])

In [63]:
res.shape

(9238, 19)

In [64]:
# Handling state variants of Democratic Party + Republican Party
res = res.replace({'DEMOCRATIC-FARMER-LABOR': 'DEMOCRAT', 'DEMOCRATIC-NONPARTISAN LEAGUE': 'DEMOCRAT', 'DEMOCRATIC-NPL': 'DEMOCRAT',
                  'REPUBLICAN, LIBERTARIAN': 'REPUBLICAN', 'WRITE-IN (DEMOCRATIC)': 'DEMOCRAT', 'WRITE-IN (REPUBLICAN)': 'REPUBLICAN'})

# Handling significant independents
res.loc[res['candidate'] == 'CARA MUND', 'party'] = 'DEMOCRAT'

In [65]:
# Focus on two party vote share in the model - no need to wrangle with third parties for now
res = res[res['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]
res.head()

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes
0,"""DAN"" LUX",2022,LOUISIANA,LA,22,72,45,US HOUSE,2,GEN,False,False,REPUBLICAN,False,TOTAL,205047,False,20250910,46927
1,A DONALD MCEACHIN,2016,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,346656,False,20250910,200136
2,A DONALD MCEACHIN,2018,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,299854,False,20250910,187642
3,A DONALD MCEACHIN,2020,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,391345,False,20250910,241142
4,A DONALD MCEACHIN,2022,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,244978,False,20250910,159044


In [66]:
# To make life easier later wrt fuzzy string matching
res = res.replace({
    'LIZZIE FLETCHER': 'ELIZABETH FLETCHER',
})

In [67]:
res_12 = res[res['year'] == 2012]
res = res[res['year'] >= cutoff_year-2]

In [68]:
res_12.shape

(839, 19)

In [69]:
data_12 = pd.pivot_table(data=res_12, values='candidatevotes', columns=['party'], index=['year', 'state', 'state_po', 'special', 'district'], aggfunc='sum').reset_index()
data_12 = data_12.fillna(0)
data_12 = data_12.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
data_12['dem_2p_pct'] = data_12['dem'] / (data_12['dem'] + data_12['rep']) * 100
data_12.to_csv('transformed/house_res_2012.csv')
data_12.head()

party,year,state,state_po,special,district,dem,rep,dem_2p_pct
0,2012,ALABAMA,AL,False,1,0.0,196374.0,0.000000
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,36.340563
2,2012,ALABAMA,AL,False,3,98141.0,175306.0,35.890319
3,2012,ALABAMA,AL,False,4,69706.0,199071.0,25.934511
4,2012,ALABAMA,AL,False,5,101772.0,189185.0,34.978365


In [70]:
res.shape

(5911, 19)

## FEC Data Wrangling

FEC data from: https://www.fec.gov/data/browse-data/?tab=bulk-data

In [71]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [72]:
webl12 = pd.read_table('data/fec/webl12.txt', sep='|', names=fec_webl_colnames)
webl14 = pd.read_table('data/fec/webl14.txt', sep='|', names=fec_webl_colnames)
webl16 = pd.read_table('data/fec/webl16.txt', sep='|', names=fec_webl_colnames)
webl18 = pd.read_table('data/fec/webl18.txt', sep='|', names=fec_webl_colnames)
webl20 = pd.read_table('data/fec/webl20.txt', sep='|', names=fec_webl_colnames)
webl22 = pd.read_table('data/fec/webl22.txt', sep='|', names=fec_webl_colnames)
webl24 = pd.read_table('data/fec/webl24.txt', sep='|', names=fec_webl_colnames)
webl24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,83969.49,0.00,0.0,0.0,0.00,0.0,0.00,10800004.13,AK,0.0,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,104330.06,599.00,0.0,0.0,25000.00,0.0,425000.00,2309088.23,AK,0.0,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,205811.99,0.00,0.0,0.0,0.00,0.0,0.00,286828.61,AK,0.0,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,410.16,10115.56,2500.0,0.0,0.00,0.0,0.00,3076.82,AL,1.0,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,69290.42,0.00,0.0,0.0,176202.71,0.0,254535.87,1063039.38,AL,1.0,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0


In [73]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

cn12 = pd.read_table('data/fec/cn12.txt', sep='|', names=fec_cn_colnames)
cn14 = pd.read_table('data/fec/cn14.txt', sep='|', names=fec_cn_colnames)
cn16 = pd.read_table('data/fec/cn16.txt', sep='|', names=fec_cn_colnames)
cn18 = pd.read_table('data/fec/cn18.txt', sep='|', names=fec_cn_colnames)
cn20 = pd.read_table('data/fec/cn20.txt', sep='|', names=fec_cn_colnames)
cn22 = pd.read_table('data/fec/cn22.txt', sep='|', names=fec_cn_colnames)
cn24 = pd.read_table('data/fec/cn24.txt', sep='|', names=fec_cn_colnames)
cn24.head()

,CAND_ID,CAND_NAME,CAND_PTY_AFFILIATION,CAND_ELECTION_YR,CAND_OFFICE_ST,CAND_OFFICE,CAND_OFFICE_DISTRICT,CAND_ICI,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H0AK00105,"LAMB, THOMAS",NNE,2020,AK,H,0.0,C,N,C00607515,1861 W LAKE LUCILLE DR,NaN,WASILLA,AK,99654
1,H0AL01055,"CARL, JERRY LEE, JR",REP,2024,AL,H,1.0,I,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685
2,H0AL01097,"AVERHART, JAMES",DEM,2024,AL,H,2.0,C,C,C00708867,811 SPRINGHILL AV,NaN,MOBILE,AL,36602
3,H0AL02087,"ROBY, MARTHA",REP,2020,AL,H,2.0,I,P,C00462143,NaN,NaN,MONTGOMERY,NaN,NaN
4,H0AL02137,"DISMUKES, WILL",REP,2020,AL,H,2.0,O,P,C00714337,PO BOX 6811188,NaN,PRATTVILLE,AL,36068


In [74]:
def wrangle_fec(webl, cn, year):
    webl['cand_name_lst'] = webl['CAND_NAME'].str.split(',')
    cn['cand_name_lst'] = cn['CAND_NAME'].str.split(',')

    def refactor_str(cand_lst):
        if len(cand_lst) == 3:
            return cand_lst[1] + ' ' + cand_lst[0] + ' ' + cand_lst[2]
        elif len(cand_lst) == 1:
            return cand_lst[0]
        else:
            return cand_lst[1] + ' ' + cand_lst[0]

    webl['cand'] = webl['cand_name_lst'].map(refactor_str)
    cn['cand'] = cn['cand_name_lst'].map(refactor_str)

    df = pd.merge(left=webl, right=cn, on='CAND_ID', how='inner')
    df = df[[col for col in df.columns.values if ('_y' not in col)]]

    df.columns = df.columns.str.strip('_x')

    df['year'] = np.full(shape=(df.shape[0],), fill_value=year)

    return df

In [75]:
fec12 = wrangle_fec(webl12, cn12, 2012)
fec14 = wrangle_fec(webl14, cn14, 2014)
fec16 = wrangle_fec(webl16, cn16, 2016)
fec18 = wrangle_fec(webl18, cn18, 2018)
fec20 = wrangle_fec(webl20, cn20, 2020)
fec22 = wrangle_fec(webl22, cn22, 2022)
fec24 = wrangle_fec(webl24, cn24, 2024)
fec24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,83969.49,0.00,0.0,0.0,0.00,0.0,0.00,10800004.13,AK,0.0,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0,"[PELTOLA, MARY]",MARY PELTOLA,2024,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501,2024
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,104330.06,599.00,0.0,0.0,25000.00,0.0,425000.00,2309088.23,AK,0.0,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0,"[BEGICH, NICHOLAS III]",NICHOLAS III BEGICH,2024,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567,2024
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,205811.99,0.00,0.0,0.0,0.00,0.0,0.00,286828.61,AK,0.0,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0,"[DAHLSTROM, NANCY]",NANCY DAHLSTROM,2024,H,C,C00856716,PO BOX 242442,NaN,ANCHORAGE,AK,99524,2024
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,410.16,10115.56,2500.0,0.0,0.00,0.0,0.00,3076.82,AL,1.0,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0,"[HOLMES, THOMAS BETHUNE MR.]",THOMAS BETHUNE MR. HOLMES,2024,H,C,C00866939,"2117 CHARINGWOOD DRIVE WEST, MOBIL",NaN,MOBILE,AL,366952916,2024
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,69290.42,0.00,0.0,0.0,176202.71,0.0,254535.87,1063039.38,AL,1.0,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0,"[CARL, JERRY LEE, JR]",JERRY LEE CARL JR,2024,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685,2024


In [76]:
fec = pd.concat([fec12, fec14, fec16, fec18, fec20, fec22, fec24], axis=0)
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H2AK00093,"URQUIDI, DOUGLAS C",C,1,DEM,3208.82,0.0,3089.84,0.00,0.0,118.98,2635.48,0.00,0.0,0.00,0.0,2520.82,573.34,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,09/30/2012,0.0,0.0,"[URQUIDI, DOUGLAS C]",DOUGLAS C URQUIDI,2012,H,N,C00515767,12134 COPPER MT DR,NaN,EAGLE RIVER,AK,99577.0,2012
1,H2AK00101,"MOORE, MATTHEW EDWARD",C,1,DEM,43184.98,0.0,43184.98,0.00,0.0,0.00,4538.56,33515.99,0.0,278.02,0.0,0.00,5130.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,12/31/2012,0.0,0.0,"[MOORE, MATTHEW EDWARD]",MATTHEW EDWARD MOORE,2012,H,C,C00520544,7035 TULUGAK CIRCLE,NaN,ANCHORAGE,AK,99507.0,2012
2,H2AK00119,"CISSNA, SHARON MARIE",C,1,DEM,19660.00,0.0,24388.00,1450.00,0.0,0.00,0.00,17842.00,0.0,0.00,0.0,13000.00,13818.00,AK,0.0,NaN,W,NaN,L,28.0,1000.0,0.0,12/31/2012,0.0,0.0,"[CISSNA, SHARON MARIE]",SHARON MARIE CISSNA,2012,H,C,NaN,2612 EAST 20TH,NaN,ANCHORAGE,AK,99508.0,2012
3,H2AK00127,"CHESNUT, DEBRA SUE",C,1,DEM,16694.00,0.0,16247.00,91.16,0.0,91.00,7700.00,0.00,0.0,0.00,0.0,0.00,8994.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,12/31/2012,2000.0,0.0,"[CHESNUT, DEBRA SUE]",DEBRA SUE CHESNUT,2012,H,C,C00523662,PO BOX 81456,NaN,FAIRBANKS,AK,99708.0,2012
4,H4AK00057,"VONDERSAAR, FRANK J",C,1,DEM,1109.00,0.0,1109.59,0.00,0.0,0.00,0.00,1100.00,0.0,84.00,0.0,0.00,9.00,AK,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,08/29/2012,0.0,0.0,"[VONDERSAAR, FRANK J]",FRANK J VONDERSAAR,2012,H,N,C00503896,1740 SALTWATER DR,NaN,HOMER,AK,996038321.0,2012


In [77]:
fec24.shape, fec.shape

((2373, 42), (16444, 42))

In [78]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'cand_name_lst', 'cand',
       'CAND_ELECTION_YR', 'CAND_OFFICE', 'CAND_STATUS', 'CAND_PCC',
       'CAND_ST1', 'CAND_ST2', 'CAND_CITY', 'CAND_ST', 'CAND_ZIP', 'year'],
      dtype=object)

In [79]:
fuzz.partial_ratio('JAMAAL BOWMAN', 'GEORGE LATIMER')

21.052631578947366

In [80]:
fuzz.token_sort_ratio('ERIC MICHAEL SWALWELL', 'ERIC SWALWELL')

76.47058823529412

In [81]:
def get_fuzzymatch_cand(year, state_po, district, party, candidate):
    '''
    :param party: Party name as noted in `res` dataframe.
    :param candidate: Candidate name as noted in the `res` dataframe.
    '''

    if party == 'DEMOCRAT':
        fec_party = 'DEM'
    elif party == 'REPUBLICAN':
        fec_party = 'REP'
    
    df = fec[(fec['year'] == year) &
        (fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district)]

    resdf = res[(res['year'] == year) &
        (res['state_po'] == state_po) &
        (res['district'] == district) &
        (res['candidate'] == candidate)]

    if party == 'DEM':
        cand_col_str = 'dem_cand'
    else:
        cand_col_str = 'rep_cand'

    # df['cand_fuzzymatch'] = df['CAND_NAME'].str.title().apply(
    #         lambda x: process.extractOne(x, mit_df[cand_col_str].values[0], scorer=fuzz.partial_ratio)[0]
    # )
    if resdf.shape[0] == 0:
        return ''

    if df.shape[0] == 0:
        #  raise ValueError('Candidate available in MIT Election Lab dataset, but not available in FEC dataset.')
        return 'Error'
    
    fuzzymatch = process.extractOne(resdf['candidate'].values[0], df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
    return fuzzymatch if fuzzymatch is None else fuzzymatch[0]

In [82]:
get_fuzzymatch_cand(2018, 'AL', 1, 'DEMOCRAT', 'ROBERT KENNEDY JR')

'KENNEDY, ROBERT JR.'

In [83]:
get_fuzzymatch_cand(2024, 'CA', 14, 'DEMOCRAT', 'ERIC SWALWELL')

'SWALWELL, ERIC MICHAEL'

In [84]:
get_fuzzymatch_cand(2024, 'CA', 12, 'DEMOCRAT', 'JENNIFER TRAN')

'TRAN, JENNIFER'

In [85]:
get_fuzzymatch_cand(2024, 'MA', 4, 'REPUBLICAN', float('nan')) # running unopposed

''

In [86]:
get_fuzzymatch_cand(2020, 'CA', 28, 'REPUBLICAN', 'ERIC EARLY')

'EARLY, ERIC'

In [87]:
res['fec_candidate_name'] = res.apply(
    lambda x: get_fuzzymatch_cand(x['year'], x['state_po'], x['district'], x['party'], x['candidate']),
    axis=1
)
res.head()

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes,fec_candidate_name
0,"""DAN"" LUX",2022,LOUISIANA,LA,22,72,45,US HOUSE,2,GEN,False,False,REPUBLICAN,False,TOTAL,205047,False,20250910,46927,"LUX, DANIEL ANTHONY"
1,A DONALD MCEACHIN,2016,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,346656,False,20250910,200136,"MCEACHIN, ASTON DONALD MR."
2,A DONALD MCEACHIN,2018,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,299854,False,20250910,187642,"MCEACHIN, ASTON DONALD"
3,A DONALD MCEACHIN,2020,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,391345,False,20250910,241142,"MCEACHIN, ASTON DONALD"
4,A DONALD MCEACHIN,2022,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,244978,False,20250910,159044,"MCEACHIN, ASTON DONALD"


In [88]:
res.shape

(5911, 20)

In [89]:
res[res['fec_candidate_name'] == 'Error']

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes,fec_candidate_name
5315,MARK S MURPHY,2012,NEW YORK,NY,36,21,13,US HOUSE,11,GEN,None,False,DEMOCRAT,False,TOTAL,214755,False,20250910,92428,Error
5595,MICHAEL G GRIMM,2012,NEW YORK,NY,36,21,13,US HOUSE,11,GEN,None,False,REPUBLICAN,False,TOTAL,214755,False,20250910,103118,Error


In [95]:
# Correct some basic discrepancies between seat numbers in FEC and MEDSL datasets for 2012
fec.loc[(fec['CAND_OFFICE_ST'] == 'NY') &
    (fec['CAND_OFFICE_DISTRICT'] == 13) &
    (fec['year'] == 2012) &
    (fec['CAND_NAME'].isin(['MURPHY, MARK', 'GRIMM, MICHAEL'])), 'CAND_OFFICE_DISTRICT'] = 11

fec.loc[(fec['CAND_OFFICE_ST'] == 'CA') &
    (fec['CAND_OFFICE_DISTRICT'] == 39) &
    (fec['year'] == 2012) &
    (fec['CAND_NAME'].isin(['SANCHEZ, LINDA'])), 'CAND_OFFICE_DISTRICT'] = 38

fec[(fec['CAND_OFFICE_ST'] == 'NY') &
    (fec['CAND_OFFICE_DISTRICT'] == 13) &
    (fec['year'] == 2012) &
    (fec['CAND_NAME'].isin(['MURPHY, MARK', 'GRIMM, MICHAEL']))]

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year


In [96]:
# Fixing some errors in the fuzzy string matching
# Below dictionary is Claude-generated because holy shit I'm not going to manually check all 5,072 rows to check for errors
corrections = {
    ('GARRY W COBB', 2014, 'NJ', 1):                        'COBB, GARY WILBERT',
    ('GARY STEIN', 2016, 'FL', 20):                         None,
    ('SCOTT W TAYLOR', 2016, 'VA', 2):                      'TAYLOR, SCOTT',
    ('JOHN BARROW', 2014, 'GA', 12):                        'BARROW, JOHN J.',
    ('JUAN VARGAS', 2016, 'CA', 51):                        'VARGAS, JUAN C.',
    ('RICHARD LIEBERMAN', 2014, 'LA', 6):                   None,
    ('ANGÉLICA MARIA DUEÑAS', 2022, 'CA', 29):              'DUENAS, ANGELICA MARIA',
    ('JOHN "JOHNNY O" OLSZEWSKI, JR.', 2024, 'MD', 2):      'OLSZEWSKI, JOHN ANTHONY JR.',
    ('JOHN R. CARTER', 2024, 'TX', 31):                     'CARTER, JOHN R REP.',
    ('CHARLES "CASPER" STOCKHAM', 2016, 'CO', 1):           'STOCKHAM, CHARLES WESLEY (CASPER)',
    ('"RICK" JOHN', 2016, 'LA', 4):                         'JOHN, RICHARD MICHAEL MR.',
    ('MICHAEL TED EVANS', 2018, 'MS', 3):                   None,
    ('MICHAEL A. RULLI', 2024, 'OH', 6):                    'RULLI, MICHAEL',
    ('MICHAEL EGGMAN', 2016, 'CA', 10):                     'EGGMAN, MICHAEL RAY',
    ('MICHAEL EGGMAN', 2014, 'CA', 10):                     'EGGMAN, MICHAEL RAY',
    ('BILL PASCRELL JR', 2016, 'NJ', 9):                    'PASCRELL, WILLIAM J. HON.',
    ('JAMES RHODES', 2020, 'KY', 1):                        None,
    ('GABBY SAUCEDO', 2014, 'AZ', 3):                       None,
    ('JEFF A DOVE JR', 2018, 'VA', 11):                     'DOVE, JEFFERY ANTHONY MR. JR.',
    ('MICHAEL GUEST', 2018, 'MS', 3):                       'GUEST, MICHAEL PATRICK',
    ('TROY A CARTER', 2022, 'LA', 2):                       'CARTER, TROY A. SR.',
    ('NICHOLAS J LALOTA', 2022, 'NY', 1):                   'NICK, LALOTA',
    ('BOB GIBBS', 2014, 'OH', 7):                           'GIBBS, ROBERT',
    ('BOB GIBBS', 2016, 'OH', 7):                           'GIBBS, ROBERT',
    ('BOB GIBBS', 2018, 'OH', 7):                           'GIBBS, ROBERT',
    ('BOB GIBBS', 2020, 'OH', 7):                           'GIBBS, ROBERT',
    ('BILL HUIZENGA', 2018, 'MI', 2):                       'HUIZENGA, WILLIAM P',
    ('BILL HUIZENGA', 2020, 'MI', 2):                       'HUIZENGA, WILLIAM P',
    ('BILL HUIZENGA', 2022, 'MI', 4):                       'HUIZENGA, WILLIAM P',
    ('BILL HUIZENGA', 2024, 'MI', 4):                       'HUIZENGA, WILLIAM P',
    ('BILL TILGHMAN', 2014, 'MD', 1):                       'TILGHMAN, WILLIAM F',
    ('BILL KREGLER', 2024, 'NY', 7):                        'KREGLER, WILLIAM',
    ('BOB MAY', 2022, 'MA', 6):                             'MAY, ROBERT',
    ('BRIAN WONNACOTT', 2014, 'UT', 3):                     'WONNACOTT, BRIAN',
    ('MARLIN A STUTZMAN', 2014, 'IN', 3):                   'STUTZMAN, MARLIN A',
    ('TIM WALBERG', 2022, 'MI', 5):                         'WALBERG, TIMOTHY L REP',
    ('THOMAS P TIFFANY', 2020, 'WI', 7):                    'TIFFANY, TOM',
    ('THOMAS P TIFFANY', 2022, 'WI', 7):                    'TIFFANY, TOM',
    ('THOMAS P. TIFFANY', 2024, 'WI', 7):                   'TIFFANY, TOM',
    ('"ANDIE" SAIZAN', 2018, 'LA', 6):                      'SAIZAN, ANDIE',
    ('"ED" TARPLEY', 2014, 'LA', 5):                        'TARPLEY, EDWARD L JR',
    ('"GUS" RANTZ', 2016, 'LA', 3):                         'RANTZ, AUGUST J IV',
    ('"JAMIE" MAYO', 2014, 'LA', 5):                        'MAYO, JAMIE',
    ('"TREY" THOMAS', 2014, 'LA', 6):                       "THOMAS, CHARLES 'TREY' DR III",
    ('JAMES "JIMMY" BEARD', 2022, 'KS', 1):                 'BEARD, JAMES KENNETH',
    ('JRMAR "JJ" JEFFERSON', 2022, 'TX', 1):                'JEFFERSON, JRMAR',
    ('HOLDEN HOGGATT', 2022, 'LA', 3):                      'HOGGATT, "HOLDEN"',
    ('DOUGLAS MACARTHUR "DOUG" MAGEE', 2014, 'MS', 3):      'MAGEE, DOUGLAS MACARTHUR (DOUG)',
    ('ROBERT LAMAR "BOB" BELL', 2014, 'LA', 6):             'BELL, ROBERT (BOB) CAPT',
    ('W A "BILL" HEDGE', 2014, 'MO', 6):                    'HEDGE, W A (BILL) DR',
    ('A J "JOHN" PETERS', 2024, 'MN', 7):                   'PETERS, ALVIN JOHN 3205946498',
    ('H D "CHIP" EVANS', 2016, 'NV', 2):                    'EVANS, HUGH D MR JR',
    ('M D "MATT" ROWE', 2016, 'VA', 1):                     'ROWE, MATTHEW',
    ('M L "MARTY" WILLIAMS', 2016, 'VA', 3):                'WILLIAMS, MARTIN L. MR.',
    ('MARK S REED', 2014, 'CA', 30):                        'REED SR., MARK STEVEN SR.',
    ('DANNY TARKANIAN', 2018, 'NV', 3):                     'TARKANIAN, DANNY',
    ('DAVID SCHWEIKERT', 2022, 'AZ', 1):                    'SCHWEIKERT, DAVID S.',
    ('GEORGE HOLDING', 2016, 'NC', 2):                      'HOLDING, GEORGE E MR.',
    ('DANIEL WEBSTER', 2016, 'FL', 11):                     'WEBSTER, DANIEL',
    ('STEVE LINDBECK', 2016, 'AK', 0):                      'LINDBECK, STEVE',
    ('MARCUS FLOWERS', 2022, 'GA', 14):                     'FLOWERS, MARCUS',
    ('STEVEN HOLDEN', 2022, 'NY', 24):                      'HOLDEN, STEVEN WESLEY SR.',
    ('LISA MCCLAIN', 2022, 'MI', 9):                        'MCCLAIN, LISA',
    ('LUIS POZZOLO', 2022, 'AZ', 7):                        'POZZOLO, LUIS B',
    ('AJA SMITH', 2022, 'CA', 39):                          'SMITH, AJA',
    ('LIZZIE PANNILL FLETCHER', 2018, 'TX', 7):             'FLETCHER, ELIZABETH',
    ('JENNIE LOU LEEDER', 2018, 'TX', 11):                  'LEEDER, VIRGINIA LOUISE',
    ('SUSAN ELLIS WILD', 2018, 'PA', 7):                    'WILD, SUSAN',
    ('MICHAEL COLE', 2016, 'TX', 14):                       'COLE, MICHAEL K',
    ('JOSH BRANNON', 2016, 'NC', 5):                        None,
    ('JOSHUA "JOSH" BRANNON', 2014, 'NC', 5):               'BRANNON, JOSHUA "JOSH" ETHAN',
    ('MICHAEL G GRIMM', 2012, 'NY', 11):     'GRIMM, MICHAEL',
    ('MARK S MURPHY', 2012, 'NY', 11):        'MURPHY, MARK',
    ('CRAIG SCHLEY', 2012, 'NY', 13):         None,
    ('RAUL R LABRADOR', 2012, 'ID', 1):       'LABRADOR, RAUL',
    ('JOHN LAFERLA', 2012, 'MD', 1):          'LA FERLA, JOHN JAMES DR',
    ('KIM VANN', 2012, 'CA', 3):              'DOLBOW VANN, KIMBERLY',
    ('BILL CASSIDY', 2012, 'LA', 6):          'CASSIDY, WILLIAM',
    ('BILL HUIZENGA', 2012, 'MI', 2):         'HUIZENGA, WILLIAM P',
    ('DAVID SANCHEZ', 2012, 'CA', 40):        None,
    ('JAMIE WALL', 2012, 'WI', 8):            'WALL, JAMES RICHARD JR',
    ('JERRY HAYDEN', 2012, 'CA', 46):         'HAYDEN, GERARD P',
    ('LINDA T SANCHEZ', 2012, 'CA', 38):      'SANCHEZ, LINDA',
    ('RANDY LOFTIN', 2012, 'CA', 5):          'LOFTIN, JOSEPH R',
    ('STEPHEN C SMITH', 2012, 'CA', 34):      None,
    ('TONY STRICKLAND', 2012, 'CA', 26):      'STRICKLAND, ANTHONY A.',

}


for (cand, year, state, dist) in corrections.keys():
    mask = (
        (res['candidate'] == cand) &
        (res['year'] == year) &
        (res['state_po'] == state) &
        (res['district'] == dist)
    )

    res.loc[mask, 'fec_candidate_name'] = corrections[(cand, year, state, dist)]

res.head()

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes,fec_candidate_name
0,"""DAN"" LUX",2022,LOUISIANA,LA,22,72,45,US HOUSE,2,GEN,False,False,REPUBLICAN,False,TOTAL,205047,False,20250910,46927,"LUX, DANIEL ANTHONY"
1,A DONALD MCEACHIN,2016,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,346656,False,20250910,200136,"MCEACHIN, ASTON DONALD MR."
2,A DONALD MCEACHIN,2018,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,299854,False,20250910,187642,"MCEACHIN, ASTON DONALD"
3,A DONALD MCEACHIN,2020,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,391345,False,20250910,241142,"MCEACHIN, ASTON DONALD"
4,A DONALD MCEACHIN,2022,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,244978,False,20250910,159044,"MCEACHIN, ASTON DONALD"


In [97]:
res.shape, fec.shape

((5911, 20), (16444, 42))

In [98]:
res_og = res.copy()

In [99]:
res = pd.merge(left=res, right=fec, left_on=['year', 'state_po', 'district', 'fec_candidate_name'], right_on=['year', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT', 'CAND_NAME'], how='left', 
               indicator=False)
res.head()

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes,fec_candidate_name,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,"""DAN"" LUX",2022,LOUISIANA,LA,22,72,45,US HOUSE,2,GEN,False,False,REPUBLICAN,False,TOTAL,205047,False,20250910,46927,"LUX, DANIEL ANTHONY",H2LA02297,"LUX, DANIEL ANTHONY",C,2.0,REP,32371.00,0.00,36052.27,0.0,100.00,-3581.27,0.0,14700.0,0.0,0.0,0.0,14700.0,17671.00,LA,2.0,NaN,NaN,NaN,NaN,NaN,0.00,0.00,11/22/2022,0.00,0.0,"[LUX, DANIEL ANTHONY]",DANIEL ANTHONY LUX,2022.0,H,C,C00821413,1104 MAPLEWOOD DR,NaN,HARVEY,LA,70058.0
1,A DONALD MCEACHIN,2016,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,346656,False,20250910,200136,"MCEACHIN, ASTON DONALD MR.",H6VA04061,"MCEACHIN, ASTON DONALD MR.",C,1.0,DEM,795103.46,0.00,516093.05,0.0,0.00,279010.41,0.0,0.0,0.0,0.0,0.0,0.0,418487.58,VA,4.0,NaN,NaN,NaN,NaN,NaN,375906.47,500.00,12/31/2016,303.03,2500.0,"[MCEACHIN, ASTON DONALD MR.]",ASTON DONALD MR. MCEACHIN,2016.0,H,C,C00610964,304 N. WILKINSON RD,NaN,HENRICO,VA,23227.0
2,A DONALD MCEACHIN,2018,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,None,False,DEMOCRAT,False,TOTAL,299854,False,20250910,187642,"MCEACHIN, ASTON DONALD",H6VA04061,"MCEACHIN, ASTON DONALD",I,1.0,DEM,863818.58,4440.43,931320.54,0.0,279010.41,211508.45,0.0,0.0,0.0,0.0,0.0,0.0,378196.56,VA,4.0,NaN,NaN,NaN,NaN,NaN,477293.61,30.75,12/31/2018,1689.77,0.0,"[MCEACHIN, ASTON DONALD]",ASTON DONALD MCEACHIN,2018.0,H,C,C00610964,PO BOX 7020,NaN,RICHMOND,VA,23221.0
3,A DONALD MCEACHIN,2020,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,391345,False,20250910,241142,"MCEACHIN, ASTON DONALD",H6VA04061,"MCEACHIN, ASTON DONALD",I,1.0,DEM,986394.45,0.00,988770.90,0.0,211508.45,209132.00,0.0,0.0,0.0,0.0,0.0,0.0,325940.51,VA,4.0,NaN,NaN,NaN,NaN,NaN,648400.00,0.00,12/31/2020,12929.01,0.0,"[MCEACHIN, ASTON DONALD]",ASTON DONALD MCEACHIN,2020.0,H,C,C00610964,NaN,NaN,RICHMOND,NaN,NaN
4,A DONALD MCEACHIN,2022,VIRGINIA,VA,51,54,40,US HOUSE,4,GEN,False,False,DEMOCRAT,False,TOTAL,244978,False,20250910,159044,"MCEACHIN, ASTON DONALD",H6VA04061,"MCEACHIN, ASTON DONALD",I,1.0,DEM,936470.02,0.00,1041249.17,0.0,209132.00,104352.85,0.0,0.0,0.0,0.0,0.0,0.0,333989.99,VA,4.0,NaN,NaN,NaN,NaN,NaN,597000.01,2000.00,12/31/2022,1325.00,0.0,"[MCEACHIN, ASTON DONALD]",ASTON DONALD MCEACHIN,2022.0,H,C,C00610964,PO BOX 7020,NaN,RICHMOND,VA,23221.0


In [100]:
res.shape

(5924, 61)

In [101]:
pd.set_option('display.max_columns', 100)

In [102]:
# Drop duplicates formed in merging
res = res.drop_duplicates(subset=['candidate', 'year', 'state_po', 'district', 'party'], keep='last', ignore_index=True)
res.shape

(5911, 61)

In [103]:
res.columns.values

array(['candidate', 'year', 'state', 'state_po', 'state_fips',
       'state_cen', 'state_ic', 'office', 'district', 'stage', 'runoff',
       'special', 'party', 'writein', 'mode', 'totalvotes', 'unofficial',
       'version', 'candidatevotes', 'fec_candidate_name', 'CAND_ID',
       'CAND_NAME', 'CAND_ICI', 'PTY_CD', 'CAND_PTY_AFFILIATION',
       'TTL_RECEIPTS', 'TRANS_FROM_AUTH', 'TTL_DISB', 'TRANS_TO_AUTH',
       'COH_BOP', 'COH_COP', 'CAND_CONTRIB', 'CAND_LOANS', 'OTHER_LOANS',
       'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY', 'DEBTS_OWED_BY',
       'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT',
       'SPEC_ELECTION', 'PRIM_ELECTION', 'RUN_ELECTION', 'GEN_ELECTION',
       'GEN_ELECTION_PRECENT', 'OTHER_POL_CMTE_CONTRIB',
       'POL_PTY_CONTRIB', 'CVG_END_DT', 'INDIV_REFUNDS', 'CMTE_REFUNDS',
       'cand_name_lst', 'cand', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',
       'CAND_ST', 'CAND_ZIP'], dtype

In [104]:
res[res.duplicated(subset=['year', 'state', 'special', 'district', 'candidate', 'TTL_INDIV_CONTRIB'])]

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes,fec_candidate_name,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP


In [105]:
res[(res['state'] == 'MINNESOTA') & (res['year'] == 2024) & (res['district'] == 3)]['CAND_ICI']

3243    O
5368    O
Name: CAND_ICI, dtype: object

In [106]:
res[(res['state'] == 'OREGON') & (res['year'] == 2024) & (res['district'] == 3)]

,candidate,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,party,writein,mode,totalvotes,unofficial,version,candidatevotes,fec_candidate_name,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,COH_COP,CAND_CONTRIB,CAND_LOANS,OTHER_LOANS,CAND_LOAN_REPAY,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS,cand_name_lst,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
2720,JOANNA HARBOUR,2024,OREGON,OR,41,92,72,US HOUSE,3,GEN,False,False,REPUBLICAN,False,TOTAL,334369,False,20250910,84344,"HARBOUR, JOANNA",H4OR03218,"HARBOUR, JOANNA",O,2.0,REP,20339.54,0.00,18970.37,0.0,0.0,1369.17,0.00,0.0,0.0,0.0,0.0,0.0,15329.54,OR,3.0,NaN,NaN,NaN,NaN,NaN,5000.00,0.0,12/31/2024,125.0,0.0,"[HARBOUR, JOANNA]",JOANNA HARBOUR,2024.0,H,C,C00870451,27812 S HWY 211,NaN,ESTACADA,OR,97023
3904,MAXINE E. DEXTER,2024,OREGON,OR,41,92,72,US HOUSE,3,GEN,False,False,DEMOCRAT,False,TOTAL,334369,False,20250910,226405,"DEXTER, MAXINE",H4OR03192,"DEXTER, MAXINE",O,1.0,DEM,1818244.55,9154.82,1795758.18,0.0,0.0,22486.37,6283.83,0.0,0.0,0.0,0.0,0.0,1530978.36,OR,3.0,NaN,NaN,NaN,NaN,NaN,267630.81,2500.0,12/31/2024,6846.0,0.0,"[DEXTER, MAXINE]",MAXINE DEXTER,2024.0,H,C,C00859108,PO BOX 12209,NaN,PORTLAND,OR,97212


In [107]:
res = res.drop_duplicates(subset=['state', 'year', 'district', 'candidate'], keep='last')
res.shape

(5911, 61)

In [108]:
res['inc'] = res['CAND_ICI'].map(lambda x: True if x == 'I' else False)
res['inc'].value_counts()

inc
False    3273
True     2638
Name: count, dtype: int64

## Pivoting Past Results

In [109]:
# aggfunc is sum to account for races where two or more candidates from the same party are running in the general,
# we sum the votes of these candidates together as part of calculating two-party vote share
data = pd.pivot_table(data=res, values='candidatevotes', columns=['party'], index=['year', 'state', 'state_po', 'special', 'district'], aggfunc='sum').reset_index()
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN
0,2012,ALABAMA,AL,False,1,NaN,196374.0
1,2012,ALABAMA,AL,False,2,103092.0,180591.0
2,2012,ALABAMA,AL,False,3,98141.0,175306.0
3,2012,ALABAMA,AL,False,4,69706.0,199071.0
4,2012,ALABAMA,AL,False,5,101772.0,189185.0


In [110]:
data.duplicated(subset=['year', 'state', 'special', 'district']).any() # np.False_ --> no duplicates

np.False_

In [111]:
# Get candidates

def get_cands(year, state, special, district, party):
    res_sorted = res.sort_values(by=['year', 'state', 'district', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['district'] == district) &
        (res_sorted['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['candidate'].values[0]
    else:
        return repr(list(df['candidate'].values))

def get_incumbency_status(year, state, special, district, party):
    res_sorted = res.sort_values(by=['year', 'state', 'district', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['district'] == district) &
        (res_sorted['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['inc'].values[0]
    else:
        return [bool(x) for x in df['inc'].values]

def get_ttl_receipts(year, state, special, district, party): # Misnomer: It's actually grabbing total individual contributions, not total receipts
    res_sorted = res.sort_values(by=['year', 'state', 'district', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['district'] == district) &
        (res_sorted['party'] == party)
    ]
    if df.shape[0] == 1:
        if np.isnan(df['TTL_INDIV_CONTRIB'].values[0]):
            return 0
        return df['TTL_INDIV_CONTRIB'].values[0]
    else:
        return [(0 if np.isnan(float(x)) else float(x)) for x in df['TTL_INDIV_CONTRIB'].values]

def get_totvotes(year, state, special, district):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special) &
        (res['district'] == district)
    ]
    return df['totalvotes'].values[0]

In [112]:
data['totalvotes'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_totvotes(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953
2,2012,ALABAMA,AL,False,3,98141.0,175306.0,273930
3,2012,ALABAMA,AL,False,4,69706.0,199071.0,269118
4,2012,ALABAMA,AL,False,5,101772.0,189185.0,291293


In [113]:
data.shape

(3047, 8)

In [114]:
get_cands(2018, 'ALABAMA', False, 1, 'DEMOCRAT') # conventional

'ROBERT KENNEDY JR'

In [115]:
get_cands(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

"['JENNIFER TRAN', 'LATEEFAH SIMON']"

In [116]:
get_incumbency_status(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

[False, False]

In [117]:
get_ttl_receipts(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

[274389.14, 1913404.94]

In [118]:
get_cands(2024, 'Massachusetts'.upper(), False, 4, 'REPUBLICAN') # running unopposed

'[]'

In [119]:
get_cands(2020, 'California'.upper(), False, 28, 'REPUBLICAN')

'ERIC EARLY'

In [120]:
def get_dem_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'DEMOCRAT')

def get_rep_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'REPUBLICAN')

def get_dem_ici(year, state, special, district):
    return get_incumbency_status(year, state, special, district, 'DEMOCRAT')

def get_rep_ici(year, state, special, district):
    return get_incumbency_status(year, state, special, district, 'REPUBLICAN')

def get_dem_receipts(year, state, special, district):
    return get_ttl_receipts(year, state, special, district, 'DEMOCRAT')

def get_rep_receipts(year, state, special, district):
    return get_ttl_receipts(year, state, special, district, 'REPUBLICAN')

In [121]:
data['dem_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676,[],JO BONNER
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953,THERESE FORD,MARTHA ROBY
2,2012,ALABAMA,AL,False,3,98141.0,175306.0,273930,JOHN ANDREW HARRIS,MIKE ROGERS
3,2012,ALABAMA,AL,False,4,69706.0,199071.0,269118,DANIEL H BOMAN,ROBERT B ADERHOLT
4,2012,ALABAMA,AL,False,5,101772.0,189185.0,291293,CHARLIE L HOLLEY,MO BROOKS


In [122]:
data['dem_inc'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_ici(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_inc'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_ici(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676,[],JO BONNER,[],True
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953,THERESE FORD,MARTHA ROBY,False,True
2,2012,ALABAMA,AL,False,3,98141.0,175306.0,273930,JOHN ANDREW HARRIS,MIKE ROGERS,False,True
3,2012,ALABAMA,AL,False,4,69706.0,199071.0,269118,DANIEL H BOMAN,ROBERT B ADERHOLT,False,True
4,2012,ALABAMA,AL,False,5,101772.0,189185.0,291293,CHARLIE L HOLLEY,MO BROOKS,False,True


In [123]:
data['dem_funds'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_receipts(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_funds'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_receipts(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676,[],JO BONNER,[],True,[],564463.0
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953,THERESE FORD,MARTHA ROBY,False,True,0,480059.23
2,2012,ALABAMA,AL,False,3,98141.0,175306.0,273930,JOHN ANDREW HARRIS,MIKE ROGERS,False,True,1744.0,474560.95
3,2012,ALABAMA,AL,False,4,69706.0,199071.0,269118,DANIEL H BOMAN,ROBERT B ADERHOLT,False,True,2400.0,534734.58
4,2012,ALABAMA,AL,False,5,101772.0,189185.0,291293,CHARLIE L HOLLEY,MO BROOKS,False,True,33555.03,426014.35


In [124]:
data = data.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
data.head()

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676,[],JO BONNER,[],True,[],564463.0
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953,THERESE FORD,MARTHA ROBY,False,True,0,480059.23
2,2012,ALABAMA,AL,False,3,98141.0,175306.0,273930,JOHN ANDREW HARRIS,MIKE ROGERS,False,True,1744.0,474560.95
3,2012,ALABAMA,AL,False,4,69706.0,199071.0,269118,DANIEL H BOMAN,ROBERT B ADERHOLT,False,True,2400.0,534734.58
4,2012,ALABAMA,AL,False,5,101772.0,189185.0,291293,CHARLIE L HOLLEY,MO BROOKS,False,True,33555.03,426014.35


In [125]:
data['2party_votes'] = data['dem'] + data['rep']
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676,[],JO BONNER,[],True,[],564463.0,NaN
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953,THERESE FORD,MARTHA ROBY,False,True,0,480059.23,283683.0


In [126]:
data['dem_cand'] = data['dem_cand'].str.title()
data['rep_cand'] = data['rep_cand'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2012,ALABAMA,AL,False,1,NaN,196374.0,200676,[],Jo Bonner,[],True,[],564463.0,NaN
1,2012,ALABAMA,AL,False,2,103092.0,180591.0,283953,Therese Ford,Martha Roby,False,True,0,480059.23,283683.0


In [127]:
data['state'] = data['state'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2012,Alabama,AL,False,1,NaN,196374.0,200676,[],Jo Bonner,[],True,[],564463.0,NaN
1,2012,Alabama,AL,False,2,103092.0,180591.0,283953,Therese Ford,Martha Roby,False,True,0,480059.23,283683.0


In [128]:
# two-party vote share
data['dem_pct_2p'] = data['dem'] / data['2party_votes'] * 100
data['rep_pct_2p'] = data['rep'] / data['2party_votes'] * 100
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p
0,2012,Alabama,AL,False,1,NaN,196374.0,200676,[],Jo Bonner,[],True,[],564463.0,NaN,NaN,NaN
1,2012,Alabama,AL,False,2,103092.0,180591.0,283953,Therese Ford,Martha Roby,False,True,0,480059.23,283683.0,36.340563,63.659437


In [129]:
data['dem_tot_funds'] = data['dem_funds'].map(
    lambda x: x if isinstance(x, float) else (np.sum(np.array(x)) if (isinstance(x, list) and len(x) > 0) else 0)
)
data['rep_tot_funds'] = data['rep_funds'].map(
    lambda x: x if isinstance(x, float) else (np.sum(np.array(x)) if (isinstance(x, list) and len(x) > 0) else 0)
)
data.head()

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds
0,2012,Alabama,AL,False,1,NaN,196374.0,200676,[],Jo Bonner,[],True,[],564463.0,NaN,NaN,NaN,0.00,564463.00
1,2012,Alabama,AL,False,2,103092.0,180591.0,283953,Therese Ford,Martha Roby,False,True,0,480059.23,283683.0,36.340563,63.659437,0.00,480059.23
2,2012,Alabama,AL,False,3,98141.0,175306.0,273930,John Andrew Harris,Mike Rogers,False,True,1744.0,474560.95,273447.0,35.890319,64.109681,1744.00,474560.95
3,2012,Alabama,AL,False,4,69706.0,199071.0,269118,Daniel H Boman,Robert B Aderholt,False,True,2400.0,534734.58,268777.0,25.934511,74.065489,2400.00,534734.58
4,2012,Alabama,AL,False,5,101772.0,189185.0,291293,Charlie L Holley,Mo Brooks,False,True,33555.03,426014.35,290957.0,34.978365,65.021635,33555.03,426014.35


In [130]:
data['dem_funds'] = data['dem_funds'].fillna(0)
data['rep_funds'] = data['rep_funds'].fillna(0)
data['tot_funds'] = data['dem_tot_funds'] + data['rep_tot_funds']
data['dem_funds_2p_pct'] = data['dem_tot_funds'] / data['tot_funds'] * 100
data['rep_funds_2p_pct'] = data['rep_tot_funds'] / data['tot_funds'] * 100
data.head(3)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2012,Alabama,AL,False,1,NaN,196374.0,200676,[],Jo Bonner,[],True,[],564463.0,NaN,NaN,NaN,0.0,564463.00,564463.00,0.000000,100.000000
1,2012,Alabama,AL,False,2,103092.0,180591.0,283953,Therese Ford,Martha Roby,False,True,0,480059.23,283683.0,36.340563,63.659437,0.0,480059.23,480059.23,0.000000,100.000000
2,2012,Alabama,AL,False,3,98141.0,175306.0,273930,John Andrew Harris,Mike Rogers,False,True,1744.0,474560.95,273447.0,35.890319,64.109681,1744.0,474560.95,476304.95,0.366152,99.633848


In [131]:
data[data['dem_funds_2p_pct'] == 50]

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct


In [132]:
data.shape

(3047, 22)

In [133]:
data.to_csv('transformed/past_house_results.csv', index_label=False)